# Unit 4.1 Exercise – Naïve Bayes Implementation
**Name:** Quinjie Benedict Capayan  
**Exercise:** Naïve Bayes (Manual + Scikit-learn)

## Dataset

In [ ]:
docs = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM"),
]

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

---
## Part 1: Manual Naïve Bayes Implementation

### Step 1: Generate a Bag of Words (Word Frequency)

In [ ]:
import re
from collections import defaultdict
import math

def tokenize(text):
    """Lowercase and extract alphabetic tokens."""
    return re.findall(r'\b[a-zA-Z]+\b', text.lower())

# Build vocabulary and per-class word counts
class_docs = defaultdict(list)
for text, label in docs:
    class_docs[label].append(tokenize(text))

vocab = set(word for text, _ in docs for word in tokenize(text))

word_count = defaultdict(lambda: defaultdict(int))
class_total = defaultdict(int)

for label, token_lists in class_docs.items():
    for tokens in token_lists:
        for word in tokens:
            word_count[label][word] += 1
            class_total[label] += 1

print(f"Vocabulary ({len(vocab)} words):")
print(sorted(vocab))
print()
for label in ["HAM", "SPAM"]:
    print(f"Bag of Words – {label}:")
    print(dict(sorted(word_count[label].items())))
    print()

### Step 2: Calculate Prior Probabilities for HAM and SPAM

In [ ]:
total_docs = len(docs)
prior = {label: len(class_docs[label]) / total_docs for label in class_docs}

print("=== Prior Probabilities ===")
for label in ["HAM", "SPAM"]:
    count = len(class_docs[label])
    print(f"P({label}) = {count}/{total_docs} = {prior[label]:.4f}")

### Step 3: Calculate Likelihood of Tokens w.r.t. Each Class
Using **Laplace (add-1) smoothing** to handle unseen words.

In [ ]:
V = len(vocab)

def likelihood(word, label):
    """P(word | label) with Laplace smoothing."""
    return (word_count[label][word] + 1) / (class_total[label] + V)

print("=== Sample Likelihoods ===")
sample_words = ["free", "meeting", "click", "office", "limited", "offer"]
print(f"{'Word':<12} {'P(word|HAM)':>14} {'P(word|SPAM)':>14}")
print("-" * 42)
for word in sample_words:
    print(f"{word:<12} {likelihood(word, 'HAM'):>14.6f} {likelihood(word, 'SPAM'):>14.6f}")

### Step 4: Classify Test Sentences

In [ ]:
def classify_manual(sentence):
    """Classify a sentence using log-probability Naïve Bayes."""
    tokens = tokenize(sentence)
    scores = {}
    for label in ["HAM", "SPAM"]:
        log_prob = math.log(prior[label])
        for token in tokens:
            log_prob += math.log(likelihood(token, label))
        scores[label] = log_prob

    predicted = max(scores, key=scores.get)
    print(f"Sentence : '{sentence}'")
    for label, score in scores.items():
        print(f"  log P({label}|doc) = {score:.4f}")
    print(f"  → Predicted class: {predicted}")
    print()
    return predicted

print("=== Manual Naïve Bayes Classification ===")
for sentence in test_sentences:
    classify_manual(sentence)

---
## Part 2: Using Scikit-learn – Multinomial Naïve Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

# Prepare training data
texts = [text for text, _ in docs]
labels = [label for _, label in docs]

# Vectorize using Bag of Words
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(texts)

# Train the classifier
clf = MultinomialNB()
clf.fit(X_train, labels)

print("Model trained successfully!")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Classes: {clf.classes_}")

In [ ]:
# Classify test sentences
X_test = vectorizer.transform(test_sentences)
predictions = clf.predict(X_test)
probabilities = clf.predict_proba(X_test)

print("=== Scikit-learn MultinomialNB Classification ===")
for sentence, pred, probs in zip(test_sentences, predictions, probabilities):
    print(f"Sentence : '{sentence}'")
    for cls, prob in zip(clf.classes_, probs):
        print(f"  P({cls}|doc) = {prob:.4f}")
    print(f"  → Predicted class: {pred}")
    print()

---
## Summary of Results

| Test Sentence | Manual NB | Scikit-learn |
|---|---|---|
| "Limited offer, click here!" | SPAM | SPAM |
| "Meeting at 2 PM with the manager." | HAM | HAM |

Both implementations agree on the predicted classes:
- **"Limited offer, click here!"** → **SPAM** (contains spam-like words: *limited*, *click*, *offer*)
- **"Meeting at 2 PM with the manager."** → **HAM** (contains ham-like words: *meeting*, *manager*)